## 4. Learning Parameter Learning, Which Estimates the Probability Distributions of a Causal Graph

### 4.1. What Is Parameter Learning?

![04](../image/04-001-EN.excalidraw.png)

![04](../image/04-002.excalidraw.png)

### 4.2. `MLE` and Conditional Probability Tables for Discrete Data

![04](../image/04-003-EN.excalidraw.png)


![04](../image/04-004.excalidraw.png)

```py
"""
# MLE result
# evidences: {x1: (0, 1, 2), x2: (0, 1)}
# variable: {y: (0, 1)}
+---------+-----------------------+-----------------------+-----------------------+
| x1      |           0           |           1           |           2           |
+---------+-----------+-----------+-----------+-----------+-----------+-----------+
| x2      |     0     |     1     |     0     |     1     |     0     |     1     |
+---------+-----------+-----------+-----------+-----------+-----------+-----------+
| y = 0   |     7     |     6     |    11     |     9     |    11     |    12     |
+---------+-----------+-----------+-----------+-----------+-----------+-----------+
| y = 1   |     7     |     7     |    10     |     3     |     5     |    12     |
+---------+-----------+-----------+-----------+-----------+-----------+-----------+

# P( y | x1, x2 )
+---------+-----------------------+-----------------------+-----------------------+
| x1      |           0           |           1           |           2           |
+---------+-----------+-----------+-----------+-----------+-----------+-----------+
| x2      |     0     |     1     |     0     |     1     |     0     |     1     |
+---------+-----------+-----------+-----------+-----------+-----------+-----------+
| y = 0   |   0.5     | 0.461538  | 0.52381   |   0.75    |  0.6875   |   0.5     |
+---------+-----------+-----------+-----------+-----------+-----------+-----------+
| y = 1   |   0.5     | 0.538462  | 0.47619   |   0.25    |  0.3125   |   0.5     |
+---------+-----------+-----------+-----------+-----------+-----------+-----------+

# Example 1: Given P( y | x1=2, x2=1), the probability that y is 0 is 0.5, and the probability that y is 1 is 0.5.
# Example 2: Given P( y | x1=0, x2=1), the probability that y is 0 is 0.46, and the probability that y is 1 is 0.53.
"""
```

In [2]:
import pandas as pd
import numpy as np

from pgmpy_tutorial.helper_functions import load_college_plans

data = load_college_plans()


In [12]:
from pgmpy.base import PDAG
from pgmpy.parameter_estimator import DiscreteMLE

pdag = PDAG()
pdag.add_edge("sex", "pe", "->")
pdag.add_edge("ses", "pe", "->")
pdag.add_edge("iq", "pe", "->")
pdag.add_edge("ses", "iq", "--")
pdag.add_edge("ses", "cp", "->")
pdag.add_edge("iq", "cp", "->")
pdag.add_edge("pe", "cp", "->")

pdag.remove_edge("ses", "iq", "--")

dag = pdag.to_dag()


In [14]:
mle_estimator = DiscreteMLE()


In [16]:
mle_estimator.fit(dag, data)


DiscreteMLE()

In [17]:
mle_estimator.parameters_


[<TabularCPD representing P(sex:2) at 0x2b60f3531d0>,
 <TabularCPD representing P(pe:2 | iq:4, ses:4, sex:2) at 0x2b60f256ad0>,
 <TabularCPD representing P(cp:2 | iq:4, pe:2, ses:4) at 0x2b60f2578a0>,
 <TabularCPD representing P(ses:4) at 0x2b61051f050>,
 <TabularCPD representing P(iq:4) at 0x2b60f354750>]

In [19]:
print(mle_estimator.parameters_[0])
print(mle_estimator.parameters_[1])
print(mle_estimator.parameters_[2])
print(mle_estimator.parameters_[3])
print(mle_estimator.parameters_[4])


+--------+----------+
| sex(1) | 0.483718 |
+--------+----------+
| sex(2) | 0.516282 |
+--------+----------+
+-------+---------------------+-----+---------------------+
| iq    | iq(1)               | ... | iq(4)               |
+-------+---------------------+-----+---------------------+
| ses   | ses(1)              | ... | ses(4)              |
+-------+---------------------+-----+---------------------+
| sex   | sex(1)              | ... | sex(2)              |
+-------+---------------------+-----+---------------------+
| pe(1) | 0.8209302325581396  | ... | 0.11923076923076924 |
+-------+---------------------+-----+---------------------+
| pe(2) | 0.17906976744186046 | ... | 0.8807692307692307  |
+-------+---------------------+-----+---------------------+
+-------+----------------------+-----+---------------------+
| iq    | iq(1)                | ... | iq(4)               |
+-------+----------------------+-----+---------------------+
| pe    | pe(1)                | ... | pe(2)   

In [23]:
mle_estimator.parameters_[1].to_dataframe()


pe                 1         2
iq ses sex                    
1  1   1    0.820930  0.179070
       2    0.896484  0.103516
   2   1    0.678261  0.321739
       2    0.766839  0.233161
   3   1    0.557692  0.442308
       2    0.611511  0.388489
   4   1    0.351351  0.648649
       2    0.373333  0.626667
2  1   1    0.672897  0.327103
       2    0.838624  0.161376
   2   1    0.566757  0.433243
       2    0.653846  0.346154
   3   1    0.406452  0.593548
       2    0.555256  0.444744
   4   1    0.196226  0.803774
       2    0.287356  0.712644
3  1   1    0.600000  0.400000
       2    0.802867  0.197133
   2   1    0.407051  0.592949
       2    0.544892  0.455108
   3   1    0.305322  0.694678
       2    0.493369  0.506631
   4   1    0.147493  0.852507
       2    0.161725  0.838275
4  1   1    0.455621  0.544379
       2    0.677019  0.322981
   2   1    0.350365  0.649635
       2    0.512000  0.488000
   3   1    0.150470  0.849530
       2    0.315625  0.684375
   4   1    0.050710  0.949290
       2    0.119231  0.880769